# Skip Options

Skip options let you **conditionally disable** parts of your pipeline based on runtime flags.

This is useful for:
- Feature flags (enable/disable processing paths)
- Environment-specific behavior (dev vs prod)
- Debugging (skip expensive operations)

In [1]:
import polars as pl

from nebula import TransformerPipeline
from nebula.storage import nebula_storage as ns
from nebula.transformers import (
    AddLiterals,
    Filter,
    RenameColumns,
    SelectColumns,
)

## Sample Data

In [2]:
# Main orders dataframe
orders_df = pl.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6],
    "customer": ["Alice", "Bob", "Charlie", "Diana", "Eve", "Frank"],
    "amount": [150.0, 80.0, 320.0, 45.0, 200.0, 95.0],
    "status": ["completed", "pending", "completed", "pending", "completed", "cancelled"],
})

print("Orders:")
print(orders_df)

Orders:
shape: (6, 4)
┌──────────┬──────────┬────────┬───────────┐
│ order_id ┆ customer ┆ amount ┆ status    │
│ ---      ┆ ---      ┆ ---    ┆ ---       │
│ i64      ┆ str      ┆ f64    ┆ str       │
╞══════════╪══════════╪════════╪═══════════╡
│ 1        ┆ Alice    ┆ 150.0  ┆ completed │
│ 2        ┆ Bob      ┆ 80.0   ┆ pending   │
│ 3        ┆ Charlie  ┆ 320.0  ┆ completed │
│ 4        ┆ Diana    ┆ 45.0   ┆ pending   │
│ 5        ┆ Eve      ┆ 200.0  ┆ completed │
│ 6        ┆ Frank    ┆ 95.0   ┆ cancelled │
└──────────┴──────────┴────────┴───────────┘


### 3.1 Skip/Perform at Pipeline Level

Use `skip=True` or `perform=False` to disable an entire pipeline.

In [3]:
# Feature flag example
ENABLE_EXPENSIVE_ENRICHMENT = False

pipe_skippable = TransformerPipeline(
    [
        AddLiterals(data=[{"alias": "enriched", "value": True}]),
        # ... expensive transformations ...
    ],
    skip=not ENABLE_EXPENSIVE_ENRICHMENT,  # Skip if feature disabled
    name="Expensive Enrichment",
)

# Alternatively use `perform`:
pipe_skippable_alt = TransformerPipeline(
    [
        AddLiterals(data=[{"alias": "enriched", "value": True}]),
    ],
    perform=ENABLE_EXPENSIVE_ENRICHMENT,  # Only run if True
    name="Expensive Enrichment",
)

print("Pipeline with skip=True returns empty:")
pipe_skippable.show()

Pipeline with skip=True returns empty:
*** Expensive Enrichment ***


In [4]:
# When skipped, pipeline acts as pass-through
result = pipe_skippable.run(orders_df)
print(f"Output unchanged when skipped: {len(result)} rows, columns: {result.columns}")

2026-03-09 12:45:50,743 | [INFO]: Starting pipeline 'Expensive Enrichment' 
2026-03-09 12:45:50,745 | [INFO]: Pipeline 'Expensive Enrichment' completed in 0.0s 


Output unchanged when skipped: 6 rows, columns: ['order_id', 'customer', 'amount', 'status']


### 3.2 Skip/Perform in Branch and Apply-to-Rows

The `branch` and `apply_to_rows` dictionaries also accept `skip` and `perform` keys.

In [5]:
ENABLE_AUDIT = False

pipe_conditional_branch = TransformerPipeline(
    [
        AddLiterals(data=[{"alias": "audit_flag", "value": "audited"}]),
    ],
    branch={
        "end": "dead-end",
        "skip": not ENABLE_AUDIT,  # Skip branch if audit disabled
    },
    name="Conditional Audit Branch",
)

# Branch is skipped, only `otherwise` runs (if provided)
result = pipe_conditional_branch.run(orders_df)
print(f"Branch skipped - output unchanged: {result.columns}")

2026-03-09 12:45:50,759 | [INFO]: Starting pipeline 'Conditional Audit Branch' 
2026-03-09 12:45:50,761 | [INFO]: Pipeline 'Conditional Audit Branch' completed in 0.0s 


Branch skipped - output unchanged: ['order_id', 'customer', 'amount', 'status']


In [6]:
# Same for apply_to_rows
APPLY_DISCOUNT = True

pipe_conditional_apply = TransformerPipeline(
    [
        AddLiterals(data=[{"alias": "has_discount", "value": True}]),
    ],
    apply_to_rows={
        "input_col": "amount",
        "operator": "ge",
        "value": 200,
        "perform": APPLY_DISCOUNT,  # Only apply if True
    },
    otherwise=[
        AddLiterals(data=[{"alias": "has_discount", "value": False}]),
    ],
    name="Conditional Discount",
)

pipe_conditional_apply.run(orders_df).sort("order_id")

2026-03-09 12:45:50,777 | [INFO]: Starting pipeline 'Conditional Discount' 
2026-03-09 12:45:50,778 | [INFO]: Entering apply_to_rows 
2026-03-09 12:45:50,799 | [INFO]: Running 'AddLiterals' ... 
2026-03-09 12:45:50,801 | [INFO]: Completed 'AddLiterals' in 0.0s 
2026-03-09 12:45:50,801 | [INFO]: Running 'AddLiterals' ... 
2026-03-09 12:45:50,803 | [INFO]: Completed 'AddLiterals' in 0.0s 
2026-03-09 12:45:50,806 | [INFO]: Pipeline 'Conditional Discount' completed in 0.0s 


order_id,customer,amount,status,has_discount
i64,str,f64,str,bool
1,"""Alice""",150.0,"""completed""",false
2,"""Bob""",80.0,"""pending""",false
3,"""Charlie""",320.0,"""completed""",true
4,"""Diana""",45.0,"""pending""",false
5,"""Eve""",200.0,"""completed""",true
6,"""Frank""",95.0,"""cancelled""",false


### 3.3 Combining Skip with Otherwise

When a branch/apply_to_rows is skipped, the `otherwise` pipeline still runs on the full dataframe.

In [7]:
pipe_skip_with_otherwise = TransformerPipeline(
    [
        # This branch is skipped
        AddLiterals(data=[{"alias": "from_branch", "value": True}]),
    ],
    branch={
        "end": "append",
        "skip": True,
    },
    otherwise=[
        # This runs on the full dataframe when branch is skipped
        AddLiterals(data=[{"alias": "from_otherwise", "value": True}]),
    ],
    name="Skip Branch, Run Otherwise",
)

result = pipe_skip_with_otherwise.run(orders_df)
print(f"Only otherwise columns: {result.columns}")
result

2026-03-09 12:45:50,825 | [INFO]: Starting pipeline 'Skip Branch, Run Otherwise' 
2026-03-09 12:45:50,825 | [INFO]: Running 'AddLiterals' ... 
2026-03-09 12:45:50,827 | [INFO]: Completed 'AddLiterals' in 0.0s 
2026-03-09 12:45:50,827 | [INFO]: Pipeline 'Skip Branch, Run Otherwise' completed in 0.0s 


Only otherwise columns: ['order_id', 'customer', 'amount', 'status', 'from_otherwise']


order_id,customer,amount,status,from_otherwise
i64,str,f64,str,bool
1,"""Alice""",150.0,"""completed""",true
2,"""Bob""",80.0,"""pending""",true
3,"""Charlie""",320.0,"""completed""",true
4,"""Diana""",45.0,"""pending""",true
5,"""Eve""",200.0,"""completed""",true
6,"""Frank""",95.0,"""cancelled""",true


---
## Summary

### Skip Options
- `skip=True` or `perform=False` → Pipeline/branch/apply is disabled
- When skipped, `otherwise` still runs (if provided)
- Works at pipeline level, branch level, and apply-to-rows level
